In [215]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, List,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage,HumanMessage
import google.genai as genai
import operator

load_dotenv()
import os

In [ ]:
client = genai.Client(
    vertexai=True, 
    project=os.getenv('GEMINI_API_KEY'), 
    location='us-central1'
)


In [217]:
class TweetState(TypedDict):
    topic : str
    tweet : str
    evaluation : Literal["approved","needs_improvement"]
    feedback : str
    iteration : int
    max_iteration : int

    tweet_history : Annotated[List[str], operator.add]
    feedback_history : Annotated[List[str], operator.add]

    


In [218]:
def generate_tweet(state:TweetState):
    # prompt - convert to string format for Gemini client
    system_content = "You are a funny and clever twitter /X influencer"
    user_content = f"""  Write a short and hilarious tweet on topic "{state['topic']}."
                     
                     Rules :
                      -Do not use question - answer format.
                      -Make it engaging and witty.
                      - Keep it under 500 characters.
                     -use observational humor and wordplay.
                     -Avoid controversial or sensitive topics.,
                     -This is version {state['iteration']} + 1

                     
                     """
    
    # Combine system and user messages
    combined_prompt = f"System: {system_content}\n\nUser: {user_content}"

    # send generator llm
    response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=combined_prompt,).text


    #return tweet
    return {'tweet': response, 'tweet_history': [response]}




In [219]:
from pydantic import BaseModel,Field

class TweetEvaluation(BaseModel):
    evaluation : Literal["approved","needs_improvement"] = Field(description="Whether the tweet is approved or needs improvement")
    feedback : str = Field(description="Constructive feedback on how to improve the tweet if needed")
    # score : int = Field(ge=0,le=5,description="A score from 0 to 5 indicating the quality of the tweet")



In [220]:


def evaluate_tweet(state:TweetState):

    #PROMPT - convert to string format
    system_content = """You are a professional tweet evaluator. You evalute tweets based on humor,originalirty,virality and tweet format."""
    user_content = f""" 
                        Evaluate the following tweet based on the criteria below:
                        Tweet: "{state['tweet']}"

                        Criteria:
                        1. Humor: Is the tweet funny and clever?
                        2. Originality: Is the tweet unique and creative?
                        3. Virality: Does the tweet have the potential to go viral?
                        4. Format: Is the tweet within the character limit and well-structured?

                        Auto-reject if :
                        - The tweet exceeds 500 characters.
                        - The tweet contains controversial or sensitive topics.
                        - The tweet is in question-answer format.
                        - Do not end with generic,throway phrases like "What do you think?" or "Let me know your thoughts."

                        ### Respond ONLY in the structured format below
                        Evaluation: <"approved" or "needs_improvement">
                        Feedback: <one paragraph explaining strengths and weaknesses of the tweeet>


                     """
    
    # Combine system and user messages
    combined_prompt = f"System: {system_content}\n\nUser: {user_content}"

    #EVALUATOR
    response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=combined_prompt,
    config={'response_mime_type': 'application/json',
        'response_schema': TweetEvaluation},
    ).parsed
    
  
    
    return {'evaluation': response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [221]:
def optimize_tweet(state:TweetState):
    #PROMPT - convert to string format
    system_content = """You are a professional tweet optimizer. You optimize tweets based on humor,originalirty,virality and tweet format."""
    user_content = f""" 
                        Optimize the following tweet based on the feedback below:
                        Tweet: "{state['tweet']}"
                        Feedback: "{state['feedback']}"

                        Rules :
                        - Keep it under 500 characters.
                        - Make it engaging and witty.
                        -use observational humor and wordplay.
                        -Avoid controversial or sensitive topics.
                        -This is version {state['iteration']} + 1

                     """
    
    # Combine system and user messages
    combined_prompt = f"System: {system_content}\n\nUser: {user_content}"

    #OPTIMIZER
    response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=combined_prompt, 
    
    ).text

    iteration = state['iteration'] + 1
    
    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [222]:
def route_evaluation(state:TweetState):
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [223]:
graph = StateGraph(TweetState)

In [224]:


graph.add_node('generate_tweet',generate_tweet)
graph.add_node('evaluate_tweet',evaluate_tweet)
graph.add_node('optimize_tweet',optimize_tweet)




graph.add_edge(START, 'generate_tweet')
graph.add_edge('generate_tweet','evaluate_tweet')

graph.add_conditional_edges('evaluate_tweet',route_evaluation, {'approved':END,'needs_improvement':'optimize_tweet'})
graph.add_edge('optimize_tweet','evaluate_tweet')






In [225]:
workflow = graph.compile()

In [226]:
#Displaying the graph
print(workflow.get_graph().draw_ascii())

            +-----------+                 
            | __start__ |                 
            +-----------+                 
                  *                       
                  *                       
                  *                       
         +----------------+               
         | generate_tweet |               
         +----------------+               
                  *                       
                  *                       
                  *                       
         +----------------+               
         | evaluate_tweet |               
         +----------------+               
           ...          **                
          .               **              
        ..                  **            
+---------+           +----------------+  
| __end__ |           | optimize_tweet |  
+---------+           +----------------+  


In [233]:
initial_state = {
    'topic': 'indian votting system',
    'iteration': 1,
    'max_iteration': 5
}

workflow.invoke(initial_state)

{'topic': 'indian votting system',
 'tweet': "Indian voting: where your finger gets inked before your future gets decided. It's like getting a participation trophy for democracy. But hey, at least you got proof you left the house. #IndiaVotes #InkcredibleIndia #DemocracyKaRang\n",
 'evaluation': 'approved',
 'feedback': 'The tweet is concise, staying within the character limit and adhering to the specified format. The humor is present through the comparison of inking fingers to participation trophies, adding a lighthearted tone. It also possesses originality by presenting a unique perspective on Indian voting, potentially resonating with a wide audience. However, the virality potential is moderate. While the content is relatable, it does not contain elements that would cause it to spread rapidly.',
 'iteration': 1,
 'max_iteration': 5,
 'tweet_history': ["Indian voting: where your finger gets inked before your future gets decided. It's like getting a participation trophy for democracy.